**API step — run locally.** LLM-as-a-Judge scoring with two independent judge models (different from the generator). Reads `results/llm/guides.jsonl`; writes `results/llm/judge_scores.csv`.

In [1]:
# 11_llm_judge_evaluation.ipynb
# API STEP -- RUN LOCALLY. LLM-as-a-Judge scoring. Reads results/llm/guides.jsonl.
# Two judge models (different from the generator) from .env: primary=LLM_MODEL_MID (gpt-5.4-mini),
# independent=LLM_MODEL_STRONG (gpt-5.4). Three personas x seven dimensions (D1-D7), temperature=0.0.
# Writes results/llm/judge_scores.csv. Requires: openai>=1.0, python-dotenv.

import os, json, time, re
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "results" / "llm"

load_dotenv(ROOT / ".env")
PRIMARY = os.getenv("LLM_MODEL_MID", "gpt-5.4-mini")
INDEPENDENT = os.getenv("LLM_MODEL_STRONG", "gpt-5.4")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

DIMS = ["D1_medical_accuracy", "D2_association_distinction", "D3_actionability",
        "D4_readability", "D5_disclaimer_adequacy", "D6_insurance_relevance", "D7_overall_quality"]

PERSONAS = {
    "medical_expert": ("a board-certified internal-medicine physician with a diabetes subspecialty, "
                       "evaluating against ADA 2024 Standards of Care including explicit thresholds "
                       "(HbA1c >= 6.5% diabetes; 5.7-6.4% prediabetes; fasting plasma glucose >= 126 mg/dL)"),
    "insurance_specialist": ("a senior actuary and health-insurance underwriting specialist, evaluating "
                             "actuarial soundness, risk-tier/premium communication utility, and compliance framing"),
    "general_public": ("a 40-year-old office worker with no medical or statistical background, evaluating "
                       "comprehensibility, realism of actions, tone, and clarity of next steps"),
}

RUBRIC = (
    "Score each dimension 1 (poor) to 5 (excellent):\n"
    "D1 Medical Accuracy; D2 Association/Causation Distinction; D3 Actionability (specific, feasible, "
    "individually tailored target values); D4 Readability & Clarity; D5 Disclaimer Adequacy "
    "('not medical advice' + professional referral); D6 Insurance/Risk Relevance; D7 Overall Quality. "
    "Judge feasibility and specificity SEPARATELY for D3: a target value alone does not guarantee feasibility. "
    "Return ONLY a JSON object with integer keys "
    '{"D1_medical_accuracy":x,"D2_association_distinction":x,"D3_actionability":x,'
    '"D4_readability":x,"D5_disclaimer_adequacy":x,"D6_insurance_relevance":x,"D7_overall_quality":x}.'
)

def parse_scores(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
        return {k: int(round(float(d[k]))) for k in DIMS if k in d}
    except Exception:
        return None

def judge(model, persona_desc, guide):
    sys = f"You are {persona_desc}. You evaluate AI-generated insurance risk consultation guides."
    user = RUBRIC + "\n\nGUIDE:\n" + guide
    resp = client.chat.completions.create(
        model=model, temperature=0.0,
        messages=[{"role": "system", "content": sys}, {"role": "user", "content": user}])
    return parse_scores(resp.choices[0].message.content)

guides = [json.loads(l) for l in open(OUT / "guides.jsonl")]
rows = []
for g in guides:
    for jname, jmodel in [("primary", PRIMARY), ("independent", INDEPENDENT)]:
        for pname, pdesc in PERSONAS.items():
            sc = None
            for _ in range(3):                        # light retry on parse failure
                sc = judge(jmodel, pdesc, g["guide"])
                if sc:
                    break
                time.sleep(0.5)
            if sc:
                rows.append({"case_id": g["case_id"], "sex": g["sex"], "method": g["method"],
                             "judge": jname, "judge_model": jmodel, "persona": pname, **sc})
            time.sleep(0.2)
    print(f"scored {g['case_id']} ({g['method']})")

df = pd.DataFrame(rows)
df.to_csv(OUT / "judge_scores.csv", index=False)
print(f"\nSaved results/llm/judge_scores.csv ({len(df)} rows). Next: run 12_quality_analysis.ipynb")

scored M00 (dice)
scored M00 (shap)
scored M01 (dice)
scored M01 (shap)
scored M02 (dice)
scored M02 (shap)
scored M03 (dice)
scored M03 (shap)
scored M04 (dice)
scored M04 (shap)
scored M05 (dice)
scored M05 (shap)
scored M06 (dice)
scored M06 (shap)
scored M07 (dice)
scored M07 (shap)
scored M08 (dice)
scored M08 (shap)
scored M09 (dice)
scored M09 (shap)
scored M10 (dice)
scored M10 (shap)
scored M11 (dice)
scored M11 (shap)
scored M12 (dice)
scored M12 (shap)
scored M13 (dice)
scored M13 (shap)
scored M14 (dice)
scored M14 (shap)
scored M15 (dice)
scored M15 (shap)
scored M16 (dice)
scored M16 (shap)
scored M17 (dice)
scored M17 (shap)
scored M18 (dice)
scored M18 (shap)
scored M19 (dice)
scored M19 (shap)
scored F00 (dice)
scored F00 (shap)
scored F01 (dice)
scored F01 (shap)
scored F02 (dice)
scored F02 (shap)
scored F03 (dice)
scored F03 (shap)
scored F04 (dice)
scored F04 (shap)
scored F05 (dice)
scored F05 (shap)
scored F06 (dice)
scored F06 (shap)
scored F07 (dice)
scored F07